In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d hsankesara/flickr-image-dataset

Dataset URL: https://www.kaggle.com/datasets/hsankesara/flickr-image-dataset
License(s): CC0-1.0
100% 8.16G/8.16G [07:57<00:00, 18.4MB/s]



In [ ]:
import zipfile
import os

with zipfile.ZipFile('flickr-image-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('flickr30k_images')

print(os.listdir('flickr30k_images'))

['flickr30k_images']


In [ ]:
import os
import subprocess

REPO_DIR = "flickr30k_entities-master"

if not os.path.exists(REPO_DIR):
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

In [ ]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image


IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"


REPO_DIR = "flickr30k_entities-master"
if not os.path.exists(REPO_DIR):
    print("Annotation repo not found. Please run the earlier script to download it.")
    exit(1)

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    import zipfile
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

print(f"Found {len(test_image_ids)} test image IDs in annotation split.")
print(f"   First 5: {test_image_ids[:5]}")


missing = []
existing = []
for img_id in test_image_ids:
    for ext in ['.jpg', '.jpeg', '.png']:
        path = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(path):
            existing.append(img_id)
            break
    else:
        missing.append(img_id)

print(f"\nFound {len(existing)} image files on disk.")
if missing:
    print(f"Missing {len(missing)} images (first 5): {missing[:5]}")
else:
    print("All test images are present!")


if existing:
    sample_ids = existing[:3]
    print("\nAttempting to load sample images...")
    for img_id in sample_ids:
        for ext in ['.jpg', '.jpeg', '.png']:
            path = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
            if os.path.exists(path):
                try:
                    img = Image.open(path).convert("RGB")
                    print(f"Loaded image {img_id} (size: {img.size})")
                except Exception as e:
                    print(f"Failed to load {img_id}: {e}")
                break


if existing:
    sample_id = existing[0]
    print(f"\nSample phrases for image {sample_id}:")
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{sample_id}.xml")
    if os.path.exists(xml_path):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        chain_to_phrase = {}
        sent_path = os.path.join(REPO_DIR, "Sentences", f"{sample_id}.txt")
        if os.path.exists(sent_path):
            with open(sent_path, 'r') as f:
                lines = f.readlines()
                for line in lines[1:]:
                    if '#' in line:
                        parts = line.strip().split('#')
                        if len(parts) == 2:
                            chain_id, phrase = parts[0], parts[1]
                            chain_to_phrase[chain_id] = phrase
        phrases = []
        for obj in root.iter('object'):
            chain_id = obj.find('name').text
            phrase = chain_to_phrase.get(chain_id, chain_id)
            phrases.append(phrase)
        for p in phrases[:3]:
            print(f"   - {p}")
    else:
        print("XML not found for this image.")



In [ ]:
import os
import re
import xml.etree.ElementTree as ET
from PIL import Image


IMAGE_DIR = "flickr30k_images"
REPO_DIR = "flickr30k_entities-master"

def parse_phrases(sent_path):
    """
    Read the Sentences file and extract a dict mapping chain_id -> phrase.
    The format after the first line is like:
        [/EN#547/people Seven climbers] ... [/EN#548/bodyparts a rock face] ...
    We extract chain_id and the phrase (the text after the slash and before ']').
    """
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase


sample_id = "1016887272"
sent_path = os.path.join(REPO_DIR, "Sentences", f"{sample_id}.txt")
phrases = parse_phrases(sent_path)
print(f"Parsed phrases for image {sample_id}:")
for cid, ph in list(phrases.items())[:5]:
    print(f"   chain {cid} -> '{ph}'")


xml_path = os.path.join(REPO_DIR, "Annotations", f"{sample_id}.xml")
if os.path.exists(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    print(f"\nObjects from XML (first 5):")
    for i, obj in enumerate(root.iter('object')):
        if i >= 5:
            break
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is not None:
            xmin = int(bbox_elem.find('xmin').text)
            ymin = int(bbox_elem.find('ymin').text)
            xmax = int(bbox_elem.find('xmax').text)
            ymax = int(bbox_elem.find('ymax').text)
            phrase = phrases.get(chain_id, chain_id)
            print(f"   chain {chain_id} -> phrase '{phrase}'  bbox: [{xmin},{ymin},{xmax},{ymax}]")



In [ ]:
import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from tqdm import tqdm
import warnings
import logging
import os
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image
import re
import random


IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"
REPO_DIR = "flickr30k_entities-master"
NUM_EVALS = 1000
SHUFFLE = True

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-large-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()
GRID_SIZE = 24


def parse_phrases(sent_path):
    """Extract chain_id -> phrase mapping from Sentences file."""
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase

if not os.path.exists(REPO_DIR):
    import subprocess
    print("Downloading official annotation repository...")
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

image_to_objects = defaultdict(list)
for img_id in test_image_ids:
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{img_id}.xml")
    if not os.path.exists(xml_path):
        continue
    try:
        tree = ET.parse(xml_path)
    except:
        continue
    root = tree.getroot()

    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    chain_to_phrase = parse_phrases(sent_path)

    for obj in root.iter('object'):
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is None:
            continue
        xmin = int(bbox_elem.find('xmin').text)
        ymin = int(bbox_elem.find('ymin').text)
        xmax = int(bbox_elem.find('xmax').text)
        ymax = int(bbox_elem.find('ymax').text)
        phrase = chain_to_phrase.get(chain_id, chain_id)
        image_to_objects[img_id].append({
            'category_name': phrase,
            'bbox': [xmin, ymin, xmax - xmin, ymax - ymin]
        })

print(f"Loaded annotations for {len(image_to_objects)} test images.")


methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
hits = {m: 0 for m in methods}
total_evaluated = 0


def extract_all_heatmaps(inputs, valid_indices, tgt_rel):
    """

    """
    heatmaps = {}

    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(**inputs)

    for h in hooks:
        h.remove()

    all_layers_attn = torch.stack(attn_maps)


    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))
    avg_attn = avg_attn[valid_indices, 1:]

    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()


    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)

    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full

    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]

    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()

    # ours
    A_global = all_layers_attn.mean(dim=(0, 1, 2))
    A_global = A_global[valid_indices, 1:]
    A_global = A_global / (A_global.max() + 1e-10)
    W_c = A_global.numpy()

    T_ds, I_ds = W_c.shape
    N = T_ds + I_ds
    W = np.zeros((N, N))
    W[:T_ds, T_ds:] = W_c
    W[T_ds:, :T_ds] = W_c.T

    W = W + 1e-3

    D = np.sum(W, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)

    K_components = 5
    V_text = eigvecs[:T_ds, 1:K_components+1]
    V_img = eigvecs[T_ds:, 1:K_components+1]
    denoised_affinity = V_text @ V_img.T

    target_affinity = np.abs(denoised_affinity[tgt_rel, :])
    hm_ds = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
    hm_ds = (hm_ds - hm_ds.min()) / (hm_ds.max() - hm_ds.min() + 1e-10)
    heatmaps["Deep-Spec (Ours)"] = hm_ds


    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []

    def forward_hook_gcam(module, inp, out):
        nonlocal activations
        activations = out[0].detach()

    def backward_hook_gcam(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()

    def forward_hook_chefer(module, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))

    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(forward_hook_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(backward_hook_gcam))

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(forward_hook_chefer))

    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()

    for h in hooks_grad:
        h.remove()

    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))

    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        heatmap_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            heatmap_chefer += (attn * grad).mean(dim=(0, 1))

        heatmap_chefer = heatmap_chefer / len(attn_maps_grad)
        heatmap_chefer = heatmap_chefer[valid_indices, 1:]
        heatmap_chefer = torch.clamp(heatmap_chefer, min=0)

        hm_chefer = heatmap_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer = hm_chefer / (hm_chefer.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))

    return heatmaps


print(f"Starting Golden Run: Large-Scale BLIP Pointing Game over {NUM_EVALS} Flickr30k Entities...\n")

if SHUFFLE:
    random.shuffle(test_image_ids)

evaluated = 0
with tqdm(total=NUM_EVALS, desc="Evaluating", unit="obj") as pbar:
    for img_id in test_image_ids:
        if evaluated >= NUM_EVALS:
            break
        if img_id not in image_to_objects:
            continue
        objects = image_to_objects[img_id]
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            continue
        orig_W, orig_H = image.size

        sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
        if not os.path.exists(sent_path):
            continue
        with open(sent_path, 'r') as f:
            caption = f.readline().strip()
        if not caption:
            continue

        target_noun = None
        for obj in objects:
            phrase = obj['category_name']
            if phrase.lower() in caption.lower():
                target_noun = phrase
                break
        if not target_noun:
            continue
        all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]

        inputs = processor(images=image, text=caption, return_tensors="pt").to(device)

        tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        special_ids = processor.tokenizer.all_special_ids
        valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]
        target_indices = []
        for i, t in enumerate(tokens):
            if i in valid_indices:
                clean_token = t.lower().replace("##", "")
                if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                    target_indices.append(i)
        if not target_indices:
            continue
        tgt_rel = [valid_indices.index(idx) for idx in target_indices]

        heatmaps = extract_all_heatmaps(inputs, valid_indices, tgt_rel)
        if heatmaps is None:
            continue

        for m_name in methods:
            hm = heatmaps[m_name]
            if hm.max() == 0:
                continue
            max_idx = np.argmax(hm)
            max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)
            rel_x = (max_x_patch + 0.5) / GRID_SIZE
            rel_y = (max_y_patch + 0.5) / GRID_SIZE
            pixel_x = rel_x * orig_W
            pixel_y = rel_y * orig_H
            hit = False
            for bbox in all_target_bboxes:
                x_min, y_min, bbox_w, bbox_h = bbox
                x_max, y_max = x_min + bbox_w, y_min + bbox_h
                if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                    hit = True
                    break
            if hit:
                hits[m_name] += 1

        evaluated += 1
        pbar.update(1)
        if evaluated % 100 == 0:
            pbar.write(f"\n--- Pointing Game Accuracy ({evaluated}/{NUM_EVALS}) ---")
            for m_name in methods:
                acc = (hits[m_name] / evaluated) * 100
                pbar.write(f"{m_name:<25} | {acc:.1f}%")
            pbar.write("-" * 50)


print("\n" + "=" * 60)
print(" FINAL FLICKR30K POINTING GAME ACCURACY (Phrase Grounding) ".center(60))
print("=" * 60)
print(f"{'Method':<25} | {'Hit Accuracy (%)':<20}")
print("-" * 60)
for m_name in methods:
    acc = (hits[m_name] / evaluated) * 100 if evaluated > 0 else 0
    prefix = ">> " if "Deep-Spec" in m_name else "   "
    print(f"{prefix}{m_name:<22} | {acc:.2f}%")
print("=" * 60)

Loading BLIP Foundation Model...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

Loaded annotations for 999 test images.
Starting Golden Run: Large-Scale BLIP Pointing Game over 1000 Flickr30k Entities...



Evaluating:  10%|█         | 100/1000 [01:45<17:32,  1.17s/obj]


--- Pointing Game Accuracy (100/1000) ---
Raw Attention             | 15.0%
Attention Rollout         | 12.0%
ViT-Grad-CAM              | 20.0%
Attn x Grad (Chefer)      | 41.0%
Deep-Spec (Ours)          | 61.0%
--------------------------------------------------


...


--- Pointing Game Accuracy (700/1000) ---
Raw Attention             | 16.1%
Attention Rollout         | 14.4%
ViT-Grad-CAM              | 22.9%
Attn x Grad (Chefer)      | 37.1%
Deep-Spec (Ours)          | 61.3%
--------------------------------------------------


Evaluating:  80%|████████  | 800/1000 [12:59<02:54,  1.15obj/s]


--- Pointing Game Accuracy (800/1000) ---
Raw Attention             | 16.2%
Attention Rollout         | 14.6%
ViT-Grad-CAM              | 22.1%
Attn x Grad (Chefer)      | 36.1%
Deep-Spec (Ours)          | 60.0%
--------------------------------------------------


Evaluating:  87%|████████▋ | 872/1000 [14:08<02:04,  1.03obj/s]


 FINAL FLICKR30K POINTING GAME ACCURACY (Phrase Grounding)  
Method                    | Hit Accuracy (%)    
------------------------------------------------------------
   Raw Attention          | 16.74%
   Attention Rollout      | 15.02%
   ViT-Grad-CAM           | 22.48%
   Attn x Grad (Chefer)   | 36.81%
>> Deep-Spec (Ours)       | 60.09%


In [ ]:


import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from tqdm import tqdm
import warnings
import logging
import os
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend (optional, for headless)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
import random


IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"
REPO_DIR = "flickr30k_entities-master"
NUM_EXCLUSIVE = 5
RANDOMIZE = True


logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-large-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()
GRID_SIZE = 24


def parse_phrases(sent_path):
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase

if not os.path.exists(REPO_DIR):
    import subprocess
    print("Downloading official annotation repository...")
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

image_to_objects = defaultdict(list)
for img_id in test_image_ids:
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{img_id}.xml")
    if not os.path.exists(xml_path):
        continue
    try:
        tree = ET.parse(xml_path)
    except:
        continue
    root = tree.getroot()
    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    chain_to_phrase = parse_phrases(sent_path)
    for obj in root.iter('object'):
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is None:
            continue
        xmin = int(bbox_elem.find('xmin').text)
        ymin = int(bbox_elem.find('ymin').text)
        xmax = int(bbox_elem.find('xmax').text)
        ymax = int(bbox_elem.find('ymax').text)
        phrase = chain_to_phrase.get(chain_id, chain_id)
        image_to_objects[img_id].append({
            'category_name': phrase,
            'bbox': [xmin, ymin, xmax - xmin, ymax - ymin]
        })

print(f"Loaded annotations for {len(image_to_objects)} test images.")


def extract_all_heatmaps(inputs, valid_indices, tgt_rel):
    """"""
    heatmaps = {}
    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())
    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))
    with torch.no_grad():
        _ = model(**inputs)
    for h in hooks:
        h.remove()
    all_layers_attn = torch.stack(attn_maps)

    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))
    avg_attn = avg_attn[valid_indices, 1:]
    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()

    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)
    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full
    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]
    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()

    A_global = all_layers_attn.mean(dim=(0, 1, 2))
    A_global = A_global[valid_indices, 1:]
    A_global = A_global / (A_global.max() + 1e-10)
    W_c = A_global.numpy()
    T_ds, I_ds = W_c.shape
    N = T_ds + I_ds
    W = np.zeros((N, N))
    W[:T_ds, T_ds:] = W_c
    W[T_ds:, :T_ds] = W_c.T
    W = W + 1e-3
    D = np.sum(W, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)
    K_components = 5
    V_text = eigvecs[:T_ds, 1:K_components+1]
    V_img = eigvecs[T_ds:, 1:K_components+1]
    denoised_affinity = V_text @ V_img.T
    target_affinity = np.abs(denoised_affinity[tgt_rel, :])
    hm_ds = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
    hm_ds = (hm_ds - hm_ds.min()) / (hm_ds.max() - hm_ds.min() + 1e-10)
    heatmaps["Deep-Spec (Ours)"] = hm_ds

    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []
    def forward_hook_gcam(module, inp, out):
        nonlocal activations
        activations = out[0].detach()
    def backward_hook_gcam(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()
    def forward_hook_chefer(module, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))
    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(forward_hook_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(backward_hook_gcam))
    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(forward_hook_chefer))
    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()
    for h in hooks_grad:
        h.remove()
    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))
    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        heatmap_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            heatmap_chefer += (attn * grad).mean(dim=(0, 1))
        heatmap_chefer = heatmap_chefer / len(attn_maps_grad)
        heatmap_chefer = heatmap_chefer[valid_indices, 1:]
        heatmap_chefer = torch.clamp(heatmap_chefer, min=0)
        hm_chefer = heatmap_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer = hm_chefer / (hm_chefer.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))
    return heatmaps


methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
exclusive_cases = []
print("Scanning for exclusive hits (Deep-Spec hits, all others miss)...")

for img_id in tqdm(test_image_ids, desc="Evaluating images"):
    if img_id not in image_to_objects:
        continue
    objects = image_to_objects[img_id]
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(candidate):
            img_path = candidate
            break
    if img_path is None:
        continue
    try:
        image = Image.open(img_path).convert("RGB")
    except:
        continue
    orig_W, orig_H = image.size
    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    if not os.path.exists(sent_path):
        continue
    with open(sent_path, 'r') as f:
        caption = f.readline().strip()
    if not caption:
        continue
    for obj in objects:
        target_noun = obj['category_name']
        if target_noun.lower() not in caption.lower():
            continue
        all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]
        inputs = processor(images=image, text=caption, return_tensors="pt").to(device)
        tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        special_ids = processor.tokenizer.all_special_ids
        valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]
        target_indices = []
        for i, t in enumerate(tokens):
            if i in valid_indices:
                clean_token = t.lower().replace("##", "")
                if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                    target_indices.append(i)
        if not target_indices:
            continue
        tgt_rel = [valid_indices.index(idx) for idx in target_indices]
        heatmaps = extract_all_heatmaps(inputs, valid_indices, tgt_rel)
        if heatmaps is None:
            continue
        current_hits = {}
        max_coords = {}
        for m_name in methods:
            hm = heatmaps[m_name]
            if hm.max() == 0:
                current_hits[m_name] = False
                max_coords[m_name] = (0, 0)
                continue
            max_idx = np.argmax(hm)
            max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)
            rel_x = (max_x_patch + 0.5) / GRID_SIZE
            rel_y = (max_y_patch + 0.5) / GRID_SIZE
            pixel_x = rel_x * orig_W
            pixel_y = rel_y * orig_H
            hit = False
            for bbox in all_target_bboxes:
                x_min, y_min, bbox_w, bbox_h = bbox
                x_max, y_max = x_min + bbox_w, y_min + bbox_h
                if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                    hit = True
                    break
            current_hits[m_name] = hit
            max_coords[m_name] = (max_x_patch, max_y_patch)
        if current_hits["Deep-Spec (Ours)"] and not any(current_hits[m] for m in methods if m != "Deep-Spec (Ours)"):
            exclusive_cases.append({
                'image': image,
                'orig_W': orig_W,
                'orig_H': orig_H,
                'target_noun': target_noun,
                'all_target_bboxes': all_target_bboxes,
                'heatmaps': heatmaps,
                'current_hits': current_hits,
                'max_coords': max_coords,
                'image_id': img_id
            })
            if NUM_EXCLUSIVE and len(exclusive_cases) >= NUM_EXCLUSIVE:
                break
    if NUM_EXCLUSIVE and len(exclusive_cases) >= NUM_EXCLUSIVE:
        break

if not exclusive_cases:
    print("No exclusive hits found. Exiting.")
    exit()

print(f"Found {len(exclusive_cases)} exclusive hits.")

if RANDOMIZE:
    random.shuffle(exclusive_cases)
if NUM_EXCLUSIVE and len(exclusive_cases) > NUM_EXCLUSIVE:
    exclusive_cases = exclusive_cases[:NUM_EXCLUSIVE]


n_rows = len(exclusive_cases)
fig, axes = plt.subplots(n_rows, 5, figsize=(25, 5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

fig.subplots_adjust(left=0.12)

for row_idx, case in enumerate(exclusive_cases):
    image = case['image']
    orig_W, orig_H = case['orig_W'], case['orig_H']
    target_noun = case['target_noun']
    all_target_bboxes = case['all_target_bboxes']
    heatmaps = case['heatmaps']
    current_hits = case['current_hits']
    max_coords = case['max_coords']
    image_id = case['image_id']

    img_resized = image.resize((GRID_SIZE * 16, GRID_SIZE * 16))
    patch_pixel_size = 16

    for col_idx, m_name in enumerate(methods):
        ax = axes[row_idx, col_idx]
        hm = heatmaps[m_name]
        hit_status = "HIT" if current_hits[m_name] else "MISS"
        mx, my = max_coords[m_name]

        ax.imshow(img_resized, alpha=0.6)
        heatmap_img = np.array(Image.fromarray((hm * 255).astype(np.uint8)).resize((GRID_SIZE * 16, GRID_SIZE * 16), resample=Image.BICUBIC))
        ax.imshow(heatmap_img, cmap='jet', alpha=0.5)

        for bbox in all_target_bboxes:
            x_min, y_min, bbox_w, bbox_h = bbox
            scaled_x = (x_min / orig_W) * (GRID_SIZE * 16)
            scaled_y = (y_min / orig_H) * (GRID_SIZE * 16)
            scaled_w = (bbox_w / orig_W) * (GRID_SIZE * 16)
            scaled_h = (bbox_h / orig_H) * (GRID_SIZE * 16)
            rect = patches.Rectangle((scaled_x, scaled_y), scaled_w, scaled_h,
                                     linewidth=2, edgecolor='g', facecolor='none', linestyle='dashed')
            ax.add_patch(rect)

        ax.plot(mx * patch_pixel_size + 8, my * patch_pixel_size + 8,
                'r*', markersize=18, markeredgecolor='white', markeredgewidth=1.5)

        color = "green" if current_hits[m_name] else "red"
        ax.set_title(f"{m_name}\n[{hit_status}]", fontsize=14, fontweight='bold', color=color)
        ax.axis('off')

    y_pos = 1 - (row_idx + 0.5) / n_rows
    fig.text(0.02, y_pos,
             f"Image {image_id}\n'{target_noun}'",
             va='center', ha='center', rotation=90,
             fontsize=12, fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))

plt.suptitle("Deep‑Spec exclusive hits on BLIP Large (Flickr30k Entities)\nGround truth boxes in green dashed, predicted max in red star",
             fontsize=22, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0.08, 0, 1, 0.98])

pdf_filename = "deep_spec_exclusive_hits_blip_large_flickr30k.pdf"
plt.savefig(pdf_filename, format='pdf', bbox_inches='tight')

plt.show()

Loading BLIP Foundation Model...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

Loaded annotations for 999 test images.
Scanning for exclusive hits (Deep-Spec hits, all others miss)...


Evaluating images:   1%|          | 10/1000 [00:39<1:04:40,  3.92s/it]


Found 5 exclusive hits.


In [ ]:
"""
Deep-Spec Large-Scale Pointing Game Benchmark — BLIP (Flickr30k Entities)
==================================================================================
Images from local Kaggle directory, annotations from BryanPlummer official repo.
Phrase parsing fixed with regex.
"""

import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from tqdm import tqdm
import warnings
import logging
import os
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image
import re
import random

IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"
REPO_DIR = "flickr30k_entities-master"
NUM_EVALS = 1000
SHUFFLE = True

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()
GRID_SIZE = 24


def parse_phrases(sent_path):
    """Extract chain_id -> phrase mapping from Sentences file."""
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:  # skip the full caption (first line)
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase

if not os.path.exists(REPO_DIR):
    import subprocess
    print("Downloading official annotation repository...")
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

image_to_objects = defaultdict(list)
for img_id in test_image_ids:
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{img_id}.xml")
    if not os.path.exists(xml_path):
        continue
    try:
        tree = ET.parse(xml_path)
    except:
        continue
    root = tree.getroot()

    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    chain_to_phrase = parse_phrases(sent_path)

    for obj in root.iter('object'):
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is None:
            continue
        xmin = int(bbox_elem.find('xmin').text)
        ymin = int(bbox_elem.find('ymin').text)
        xmax = int(bbox_elem.find('xmax').text)
        ymax = int(bbox_elem.find('ymax').text)
        phrase = chain_to_phrase.get(chain_id, chain_id)
        image_to_objects[img_id].append({
            'category_name': phrase,
            'bbox': [xmin, ymin, xmax - xmin, ymax - ymin]
        })

print(f"Loaded annotations for {len(image_to_objects)} test images.")


methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
hits = {m: 0 for m in methods}
total_evaluated = 0


def extract_all_heatmaps(inputs, valid_indices, tgt_rel):
    """

    """
    heatmaps = {}

    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(**inputs)

    for h in hooks:
        h.remove()

    all_layers_attn = torch.stack(attn_maps)


    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))
    avg_attn = avg_attn[valid_indices, 1:]

    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()


    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)

    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full

    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]

    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()


    A_global = all_layers_attn.mean(dim=(0, 1, 2))
    A_global = A_global[valid_indices, 1:]
    A_global = A_global / (A_global.max() + 1e-10)
    W_c = A_global.numpy()

    T_ds, I_ds = W_c.shape
    N = T_ds + I_ds
    W = np.zeros((N, N))
    W[:T_ds, T_ds:] = W_c
    W[T_ds:, :T_ds] = W_c.T

    W = W + 1e-3

    D = np.sum(W, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)

    K_components = 5
    V_text = eigvecs[:T_ds, 1:K_components+1]
    V_img = eigvecs[T_ds:, 1:K_components+1]
    denoised_affinity = V_text @ V_img.T

    target_affinity = np.abs(denoised_affinity[tgt_rel, :])
    hm_ds = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
    hm_ds = (hm_ds - hm_ds.min()) / (hm_ds.max() - hm_ds.min() + 1e-10)
    heatmaps["Deep-Spec (Ours)"] = hm_ds


    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []

    def forward_hook_gcam(module, inp, out):
        nonlocal activations
        activations = out[0].detach()

    def backward_hook_gcam(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()

    def forward_hook_chefer(module, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))

    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(forward_hook_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(backward_hook_gcam))

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(forward_hook_chefer))

    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()

    for h in hooks_grad:
        h.remove()

    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))

    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        heatmap_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            heatmap_chefer += (attn * grad).mean(dim=(0, 1))

        heatmap_chefer = heatmap_chefer / len(attn_maps_grad)
        heatmap_chefer = heatmap_chefer[valid_indices, 1:]
        heatmap_chefer = torch.clamp(heatmap_chefer, min=0)

        hm_chefer = heatmap_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer = hm_chefer / (hm_chefer.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))

    return heatmaps


print(f"Starting Golden Run: base-Scale BLIP Pointing Game over {NUM_EVALS} Flickr30k Entities...\n")

if SHUFFLE:
    random.shuffle(test_image_ids)

evaluated = 0
with tqdm(total=NUM_EVALS, desc="Evaluating", unit="obj") as pbar:
    for img_id in test_image_ids:
        if evaluated >= NUM_EVALS:
            break
        if img_id not in image_to_objects:
            continue
        objects = image_to_objects[img_id]
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            continue
        orig_W, orig_H = image.size

        sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
        if not os.path.exists(sent_path):
            continue
        with open(sent_path, 'r') as f:
            caption = f.readline().strip()
        if not caption:
            continue

        target_noun = None
        for obj in objects:
            phrase = obj['category_name']
            if phrase.lower() in caption.lower():
                target_noun = phrase
                break
        if not target_noun:
            continue
        all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]

        inputs = processor(images=image, text=caption, return_tensors="pt").to(device)

        tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        special_ids = processor.tokenizer.all_special_ids
        valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]
        target_indices = []
        for i, t in enumerate(tokens):
            if i in valid_indices:
                clean_token = t.lower().replace("##", "")
                if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                    target_indices.append(i)
        if not target_indices:
            continue
        tgt_rel = [valid_indices.index(idx) for idx in target_indices]

        heatmaps = extract_all_heatmaps(inputs, valid_indices, tgt_rel)
        if heatmaps is None:
            continue

        for m_name in methods:
            hm = heatmaps[m_name]
            if hm.max() == 0:
                continue
            max_idx = np.argmax(hm)
            max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)
            rel_x = (max_x_patch + 0.5) / GRID_SIZE
            rel_y = (max_y_patch + 0.5) / GRID_SIZE
            pixel_x = rel_x * orig_W
            pixel_y = rel_y * orig_H
            hit = False
            for bbox in all_target_bboxes:
                x_min, y_min, bbox_w, bbox_h = bbox
                x_max, y_max = x_min + bbox_w, y_min + bbox_h
                if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                    hit = True
                    break
            if hit:
                hits[m_name] += 1

        evaluated += 1
        pbar.update(1)
        if evaluated % 100 == 0:
            pbar.write(f"\n--- Pointing Game Accuracy ({evaluated}/{NUM_EVALS}) ---")
            for m_name in methods:
                acc = (hits[m_name] / evaluated) * 100
                pbar.write(f"{m_name:<25} | {acc:.1f}%")
            pbar.write("-" * 50)


print("\n" + "=" * 60)
print(" FINAL FLICKR30K POINTING GAME ACCURACY (Phrase Grounding) ".center(60))
print("=" * 60)
print(f"{'Method':<25} | {'Hit Accuracy (%)':<20}")
print("-" * 60)
for m_name in methods:
    acc = (hits[m_name] / evaluated) * 100 if evaluated > 0 else 0
    prefix = ">> " if "Deep-Spec" in m_name else "   "
    print(f"{prefix}{m_name:<22} | {acc:.2f}%")
print("=" * 60)

Loading BLIP Foundation Model...


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...


--- Pointing Game Accuracy (700/1000) ---
Raw Attention             | 11.4%
Attention Rollout         | 11.4%
ViT-Grad-CAM              | 11.7%
Attn x Grad (Chefer)      | 44.6%
Deep-Spec (Ours)          | 58.1%
--------------------------------------------------



Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]

Evaluating:  80%|████████  | 800/1000 [08:15<02:10,  1.53obj/s]


--- Pointing Game Accuracy (800/1000) ---
Raw Attention             | 11.8%
Attention Rollout         | 11.8%
ViT-Grad-CAM              | 11.6%
Attn x Grad (Chefer)      | 43.8%
Deep-Spec (Ours)          | 57.9%
--------------------------------------------------



Evaluating:  87%|████████▋ | 872/1000 [08:57<01:18,  1.62obj/s]


 FINAL FLICKR30K POINTING GAME ACCURACY (Phrase Grounding)  
Method                    | Hit Accuracy (%)    
------------------------------------------------------------
   Raw Attention          | 11.70%
   Attention Rollout      | 11.70%
   ViT-Grad-CAM           | 11.58%
   Attn x Grad (Chefer)   | 43.00%
>> Deep-Spec (Ours)       | 57.91%


In [ ]:
#!/usr/bin/env python3
"""
Deep-Spec Exclusive Hits on BLIP base (Flickr30k Entities)
============================================================
Generates a grid visualisation for cases where Deep-Spec correctly points
to the target object while ALL other methods miss.

Output is saved as a PDF file.
Images are from local directory, annotations from BryanPlummer repo.
"""

import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from tqdm import tqdm
import warnings
import logging
import os
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend (optional, for headless)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
import random

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------
IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"          # folder containing the .jpg files
REPO_DIR = "flickr30k_entities-master"   # extracted annotation repo
NUM_EXCLUSIVE = 5                        # number of exclusive hits to show
RANDOMIZE = True                         # randomise selection

# ------------------------------------------------------------
# 1. SET UP MODEL
# ------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-base-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()
GRID_SIZE = 24

# ------------------------------------------------------------
# 2. LOAD ANNOTATIONS (BryanPlummer repo)
# ------------------------------------------------------------
def parse_phrases(sent_path):
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase

if not os.path.exists(REPO_DIR):
    import subprocess
    print("Downloading official annotation repository...")
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

image_to_objects = defaultdict(list)
for img_id in test_image_ids:
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{img_id}.xml")
    if not os.path.exists(xml_path):
        continue
    try:
        tree = ET.parse(xml_path)
    except:
        continue
    root = tree.getroot()
    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    chain_to_phrase = parse_phrases(sent_path)
    for obj in root.iter('object'):
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is None:
            continue
        xmin = int(bbox_elem.find('xmin').text)
        ymin = int(bbox_elem.find('ymin').text)
        xmax = int(bbox_elem.find('xmax').text)
        ymax = int(bbox_elem.find('ymax').text)
        phrase = chain_to_phrase.get(chain_id, chain_id)
        image_to_objects[img_id].append({
            'category_name': phrase,
            'bbox': [xmin, ymin, xmax - xmin, ymax - ymin]
        })

print(f"Loaded annotations for {len(image_to_objects)} test images.")

# ------------------------------------------------------------
# 3. YOUR extract_all_heatmaps() – UNCHANGED
# ------------------------------------------------------------
def extract_all_heatmaps(inputs, valid_indices, tgt_rel):
    heatmaps = {}
    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())
    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))
    with torch.no_grad():
        _ = model(**inputs)
    for h in hooks:
        h.remove()
    all_layers_attn = torch.stack(attn_maps)

    # RAW ATTENTION
    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))
    avg_attn = avg_attn[valid_indices, 1:]
    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()

    # ATTENTION ROLLOUT
    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)
    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full
    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]
    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()

    # DEEP-SPEC
    A_global = all_layers_attn.mean(dim=(0, 1, 2))
    A_global = A_global[valid_indices, 1:]
    A_global = A_global / (A_global.max() + 1e-10)
    W_c = A_global.numpy()
    T_ds, I_ds = W_c.shape
    N = T_ds + I_ds
    W = np.zeros((N, N))
    W[:T_ds, T_ds:] = W_c
    W[T_ds:, :T_ds] = W_c.T
    W = W + 1e-3
    D = np.sum(W, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)
    K_components = 5
    V_text = eigvecs[:T_ds, 1:K_components+1]
    V_img = eigvecs[T_ds:, 1:K_components+1]
    denoised_affinity = V_text @ V_img.T
    target_affinity = np.abs(denoised_affinity[tgt_rel, :])
    hm_ds = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
    hm_ds = (hm_ds - hm_ds.min()) / (hm_ds.max() - hm_ds.min() + 1e-10)
    heatmaps["Deep-Spec (Ours)"] = hm_ds

    # GRADIENT METHODS
    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []
    def forward_hook_gcam(module, inp, out):
        nonlocal activations
        activations = out[0].detach()
    def backward_hook_gcam(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()
    def forward_hook_chefer(module, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))
    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(forward_hook_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(backward_hook_gcam))
    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(forward_hook_chefer))
    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()
    for h in hooks_grad:
        h.remove()
    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))
    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        heatmap_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            heatmap_chefer += (attn * grad).mean(dim=(0, 1))
        heatmap_chefer = heatmap_chefer / len(attn_maps_grad)
        heatmap_chefer = heatmap_chefer[valid_indices, 1:]
        heatmap_chefer = torch.clamp(heatmap_chefer, min=0)
        hm_chefer = heatmap_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer = hm_chefer / (hm_chefer.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))
    return heatmaps

# ------------------------------------------------------------
# 4. COLLECT EXCLUSIVE HITS
# ------------------------------------------------------------
methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
exclusive_cases = []
print("Scanning for exclusive hits (Deep-Spec hits, all others miss)...")

for img_id in tqdm(test_image_ids, desc="Evaluating images"):
    if img_id not in image_to_objects:
        continue
    objects = image_to_objects[img_id]
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(candidate):
            img_path = candidate
            break
    if img_path is None:
        continue
    try:
        image = Image.open(img_path).convert("RGB")
    except:
        continue
    orig_W, orig_H = image.size
    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    if not os.path.exists(sent_path):
        continue
    with open(sent_path, 'r') as f:
        caption = f.readline().strip()
    if not caption:
        continue
    for obj in objects:
        target_noun = obj['category_name']
        if target_noun.lower() not in caption.lower():
            continue
        all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]
        inputs = processor(images=image, text=caption, return_tensors="pt").to(device)
        tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        special_ids = processor.tokenizer.all_special_ids
        valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]
        target_indices = []
        for i, t in enumerate(tokens):
            if i in valid_indices:
                clean_token = t.lower().replace("##", "")
                if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                    target_indices.append(i)
        if not target_indices:
            continue
        tgt_rel = [valid_indices.index(idx) for idx in target_indices]
        heatmaps = extract_all_heatmaps(inputs, valid_indices, tgt_rel)
        if heatmaps is None:
            continue
        current_hits = {}
        max_coords = {}
        for m_name in methods:
            hm = heatmaps[m_name]
            if hm.max() == 0:
                current_hits[m_name] = False
                max_coords[m_name] = (0, 0)
                continue
            max_idx = np.argmax(hm)
            max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)
            rel_x = (max_x_patch + 0.5) / GRID_SIZE
            rel_y = (max_y_patch + 0.5) / GRID_SIZE
            pixel_x = rel_x * orig_W
            pixel_y = rel_y * orig_H
            hit = False
            for bbox in all_target_bboxes:
                x_min, y_min, bbox_w, bbox_h = bbox
                x_max, y_max = x_min + bbox_w, y_min + bbox_h
                if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                    hit = True
                    break
            current_hits[m_name] = hit
            max_coords[m_name] = (max_x_patch, max_y_patch)
        if current_hits["Deep-Spec (Ours)"] and not any(current_hits[m] for m in methods if m != "Deep-Spec (Ours)"):
            exclusive_cases.append({
                'image': image,
                'orig_W': orig_W,
                'orig_H': orig_H,
                'target_noun': target_noun,
                'all_target_bboxes': all_target_bboxes,
                'heatmaps': heatmaps,
                'current_hits': current_hits,
                'max_coords': max_coords,
                'image_id': img_id
            })
            if NUM_EXCLUSIVE and len(exclusive_cases) >= NUM_EXCLUSIVE:
                break
    if NUM_EXCLUSIVE and len(exclusive_cases) >= NUM_EXCLUSIVE:
        break

if not exclusive_cases:
    print("No exclusive hits found. Exiting.")
    exit()

print(f"Found {len(exclusive_cases)} exclusive hits.")

if RANDOMIZE:
    random.shuffle(exclusive_cases)
if NUM_EXCLUSIVE and len(exclusive_cases) > NUM_EXCLUSIVE:
    exclusive_cases = exclusive_cases[:NUM_EXCLUSIVE]

# ------------------------------------------------------------
# 5. GENERATE GRID FIGURE – SAVED AS PDF
# ------------------------------------------------------------
n_rows = len(exclusive_cases)
fig, axes = plt.subplots(n_rows, 5, figsize=(25, 5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

fig.subplots_adjust(left=0.12)

for row_idx, case in enumerate(exclusive_cases):
    image = case['image']
    orig_W, orig_H = case['orig_W'], case['orig_H']
    target_noun = case['target_noun']
    all_target_bboxes = case['all_target_bboxes']
    heatmaps = case['heatmaps']
    current_hits = case['current_hits']
    max_coords = case['max_coords']
    image_id = case['image_id']

    img_resized = image.resize((GRID_SIZE * 16, GRID_SIZE * 16))
    patch_pixel_size = 16

    for col_idx, m_name in enumerate(methods):
        ax = axes[row_idx, col_idx]
        hm = heatmaps[m_name]
        hit_status = "HIT" if current_hits[m_name] else "MISS"
        mx, my = max_coords[m_name]

        ax.imshow(img_resized, alpha=0.6)
        heatmap_img = np.array(Image.fromarray((hm * 255).astype(np.uint8)).resize((GRID_SIZE * 16, GRID_SIZE * 16), resample=Image.BICUBIC))
        ax.imshow(heatmap_img, cmap='jet', alpha=0.5)

        for bbox in all_target_bboxes:
            x_min, y_min, bbox_w, bbox_h = bbox
            scaled_x = (x_min / orig_W) * (GRID_SIZE * 16)
            scaled_y = (y_min / orig_H) * (GRID_SIZE * 16)
            scaled_w = (bbox_w / orig_W) * (GRID_SIZE * 16)
            scaled_h = (bbox_h / orig_H) * (GRID_SIZE * 16)
            rect = patches.Rectangle((scaled_x, scaled_y), scaled_w, scaled_h,
                                     linewidth=2, edgecolor='g', facecolor='none', linestyle='dashed')
            ax.add_patch(rect)

        ax.plot(mx * patch_pixel_size + 8, my * patch_pixel_size + 8,
                'r*', markersize=18, markeredgecolor='white', markeredgewidth=1.5)

        color = "green" if current_hits[m_name] else "red"
        ax.set_title(f"{m_name}\n[{hit_status}]", fontsize=14, fontweight='bold', color=color)
        ax.axis('off')

    y_pos = 1 - (row_idx + 0.5) / n_rows
    fig.text(0.02, y_pos,
             f"Image {image_id}\n'{target_noun}'",
             va='center', ha='center', rotation=90,
             fontsize=12, fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))

plt.suptitle("Deep‑Spec exclusive hits on BLIP base (Flickr30k Entities)\nGround truth boxes in green dashed, predicted max in red star",
             fontsize=22, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0.08, 0, 1, 0.98])

pdf_filename = "deep_spec_exclusive_hits_blip_base_flickr30k.pdf"
plt.savefig(pdf_filename, format='pdf', bbox_inches='tight')

plt.show()

Loading BLIP Foundation Model...


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Loaded annotations for 999 test images.
Scanning for exclusive hits (Deep-Spec hits, all others miss)...


Evaluating images:   1%|          | 10/1000 [00:18<31:13,  1.89s/it]


Found 5 exclusive hits.


In [ ]:
"""
Deep-Spec Large-Scale Pointing Game Benchmark — BLIP (Flickr30k Entities)
==================================================================================
Images from local Kaggle directory, annotations from BryanPlummer official repo.
Phrase parsing fixed with regex.
"""

import torch
import numpy as np
from scipy.linalg import eigh
from transformers import BlipProcessor, BlipForImageTextRetrieval
from tqdm import tqdm
import warnings
import logging
import os
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from PIL import Image
import re
import random


IMAGE_DIR = "/content/flickr30k_images/flickr30k_images/flickr30k_images"
REPO_DIR = "flickr30k_entities-master"
NUM_EVALS = 1000
SHUFFLE = True

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BLIP Foundation Model...")
MODEL_NAME = "Salesforce/blip-itm-large-coco"
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()
GRID_SIZE = 24


def parse_phrases(sent_path):
    """Extract chain_id -> phrase mapping from Sentences file."""
    chain_to_phrase = {}
    if not os.path.exists(sent_path):
        return chain_to_phrase
    with open(sent_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:  # skip the full caption (first line)
            pattern = r'\[/EN#(\d+)/[^ ]+ (.*?)\]'
            matches = re.findall(pattern, line)
            for chain_id, phrase in matches:
                chain_to_phrase[chain_id] = phrase.strip()
    return chain_to_phrase

if not os.path.exists(REPO_DIR):
    import subprocess
    print("Downloading official annotation repository...")
    subprocess.run(["wget", "-O", "master.zip", "https://github.com/BryanPlummer/flickr30k_entities/archive/refs/heads/master.zip"], check=True)
    subprocess.run(["unzip", "-o", "master.zip"], check=True)
    os.remove("master.zip")

ANNO_ZIP = os.path.join(REPO_DIR, "annotations.zip")
if os.path.exists(ANNO_ZIP) and not os.path.exists(os.path.join(REPO_DIR, "Annotations")):
    print("Extracting annotations.zip...")
    with zipfile.ZipFile(ANNO_ZIP, 'r') as zip_ref:
        zip_ref.extractall(REPO_DIR)

with open(os.path.join(REPO_DIR, "test.txt"), "r") as f:
    test_image_ids = [line.strip().split('.')[0] for line in f]

image_to_objects = defaultdict(list)
for img_id in test_image_ids:
    xml_path = os.path.join(REPO_DIR, "Annotations", f"{img_id}.xml")
    if not os.path.exists(xml_path):
        continue
    try:
        tree = ET.parse(xml_path)
    except:
        continue
    root = tree.getroot()

    # Get chain -> phrase mapping
    sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
    chain_to_phrase = parse_phrases(sent_path)

    # Parse XML objects
    for obj in root.iter('object'):
        chain_id = obj.find('name').text
        bbox_elem = obj.find('bndbox')
        if bbox_elem is None:
            continue
        xmin = int(bbox_elem.find('xmin').text)
        ymin = int(bbox_elem.find('ymin').text)
        xmax = int(bbox_elem.find('xmax').text)
        ymax = int(bbox_elem.find('ymax').text)
        phrase = chain_to_phrase.get(chain_id, chain_id)  # fallback to chain ID if not found
        image_to_objects[img_id].append({
            'category_name': phrase,
            'bbox': [xmin, ymin, xmax - xmin, ymax - ymin]
        })

print(f"Loaded annotations for {len(image_to_objects)} test images.")


methods = ["Raw Attention", "Attention Rollout", "ViT-Grad-CAM", "Attn x Grad (Chefer)", "Deep-Spec (Ours)"]
hits = {m: 0 for m in methods}
results_per_sample = {m: [] for m in methods}   # NEW: stores 0/1 hit outcome per evaluated sample
total_evaluated = 0


def extract_all_heatmaps(inputs, valid_indices, tgt_rel):
    """
    Extracted EXACTLY using the author's provided single-image math.
    (Untouched)
    """
    heatmaps = {}


    attn_maps = []
    hooks = []
    def hook_fn(module, inp, out):
        attn_maps.append(inp[0].detach().cpu())

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks.append(module.register_forward_hook(hook_fn))

    with torch.no_grad():
        _ = model(**inputs)

    for h in hooks:
        h.remove()

    all_layers_attn = torch.stack(attn_maps)


    avg_attn = all_layers_attn.mean(dim=(0, 1, 2))
    avg_attn = avg_attn[valid_indices, 1:]

    hm_raw = avg_attn[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_raw = hm_raw / (hm_raw.max() + 1e-10)
    heatmaps["Raw Attention"] = hm_raw.numpy()


    T, I = all_layers_attn.shape[-2], all_layers_attn.shape[-1]
    rollout = torch.eye(T + I)

    for layer_attn in all_layers_attn:
        avg_heads = layer_attn.mean(dim=(0, 1))
        full = torch.zeros((T + I, T + I))
        full[:T, T:] = avg_heads
        full[T:, :T] = avg_heads.T
        full = 0.5 * full + 0.5 * torch.eye(T + I)
        rollout = rollout @ full

    text_to_img_rollout = rollout[:T, T:]
    filtered_rollout = text_to_img_rollout[valid_indices, 1:]

    hm_roll = filtered_rollout[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
    hm_roll = hm_roll / (hm_roll.max() + 1e-10)
    heatmaps["Attention Rollout"] = hm_roll.numpy()

    # deep-spec
    A_global = all_layers_attn.mean(dim=(0, 1, 2))
    A_global = A_global[valid_indices, 1:]
    A_global = A_global / (A_global.max() + 1e-10)
    W_c = A_global.numpy()

    T_ds, I_ds = W_c.shape
    N = T_ds + I_ds
    W = np.zeros((N, N))
    W[:T_ds, T_ds:] = W_c
    W[T_ds:, :T_ds] = W_c.T

    W = W + 1e-3

    D = np.sum(W, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    L_sym = np.eye(N) - D_inv_sqrt @ W @ D_inv_sqrt
    _, eigvecs = eigh(L_sym)

    K_components = 5
    V_text = eigvecs[:T_ds, 1:K_components+1]
    V_img = eigvecs[T_ds:, 1:K_components+1]
    denoised_affinity = V_text @ V_img.T

    target_affinity = np.abs(denoised_affinity[tgt_rel, :])
    hm_ds = target_affinity.max(axis=0).reshape(GRID_SIZE, GRID_SIZE)
    hm_ds = (hm_ds - hm_ds.min()) / (hm_ds.max() - hm_ds.min() + 1e-10)
    heatmaps["Deep-Spec (Ours)"] = hm_ds


    activations, gradients = None, None
    attn_maps_grad, grad_maps = [], []

    def forward_hook_gcam(module, inp, out):
        nonlocal activations
        activations = out[0].detach()

    def backward_hook_gcam(module, grad_in, grad_out):
        nonlocal gradients
        gradients = grad_out[0].detach()

    def forward_hook_chefer(module, inp, out):
        attn = inp[0]
        attn_maps_grad.append(attn)
        attn.retain_grad()
        attn.register_hook(lambda g: grad_maps.append(g))

    hooks_grad = []
    target_layer = model.vision_model.encoder.layers[-1]
    hooks_grad.append(target_layer.register_forward_hook(forward_hook_gcam))
    hooks_grad.append(target_layer.register_full_backward_hook(backward_hook_gcam))

    for name, module in model.named_modules():
        if "crossattention.self.dropout" in name:
            hooks_grad.append(module.register_forward_hook(forward_hook_chefer))

    model.zero_grad()
    outputs = model(**inputs)
    itm_score = outputs.itm_score[0, 1]
    itm_score.backward()

    for h in hooks_grad:
        h.remove()

    if gradients is not None and activations is not None:
        weights = gradients.mean(dim=(0, 1))
        cam = (weights * activations).sum(dim=-1).squeeze(0)
        cam = torch.relu(cam)[1:]
        cam = cam.reshape(GRID_SIZE, GRID_SIZE)
        cam = cam / (cam.max() + 1e-10)
        heatmaps["ViT-Grad-CAM"] = cam.detach().cpu().numpy()
    else:
        heatmaps["ViT-Grad-CAM"] = np.zeros((GRID_SIZE, GRID_SIZE))

    if len(attn_maps_grad) > 0 and len(grad_maps) > 0:
        heatmap_chefer = torch.zeros_like(attn_maps_grad[0].mean(dim=(0, 1)))
        for attn, grad in zip(attn_maps_grad, grad_maps):
            heatmap_chefer += (attn * grad).mean(dim=(0, 1))

        heatmap_chefer = heatmap_chefer / len(attn_maps_grad)
        heatmap_chefer = heatmap_chefer[valid_indices, 1:]
        heatmap_chefer = torch.clamp(heatmap_chefer, min=0)

        hm_chefer = heatmap_chefer[tgt_rel].max(dim=0).values.reshape(GRID_SIZE, GRID_SIZE)
        hm_chefer = hm_chefer / (hm_chefer.max() + 1e-10)
        heatmaps["Attn x Grad (Chefer)"] = hm_chefer.detach().cpu().numpy()
    else:
        heatmaps["Attn x Grad (Chefer)"] = np.zeros((GRID_SIZE, GRID_SIZE))

    return heatmaps


print(f"Starting Golden Run: Large-Scale BLIP LArge Pointing Game over {NUM_EVALS} Flickr30k Entities...\n")

if SHUFFLE:
    random.shuffle(test_image_ids)

evaluated = 0
with tqdm(total=NUM_EVALS, desc="Evaluating", unit="obj") as pbar:
    for img_id in test_image_ids:
        if evaluated >= NUM_EVALS:
            break
        if img_id not in image_to_objects:
            continue
        objects = image_to_objects[img_id]
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png']:
            candidate = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            continue
        orig_W, orig_H = image.size

        sent_path = os.path.join(REPO_DIR, "Sentences", f"{img_id}.txt")
        if not os.path.exists(sent_path):
            continue
        with open(sent_path, 'r') as f:
            caption = f.readline().strip()
        if not caption:
            continue

        target_noun = None
        for obj in objects:
            phrase = obj['category_name']
            if phrase.lower() in caption.lower():
                target_noun = phrase
                break
        if not target_noun:
            continue
        all_target_bboxes = [obj['bbox'] for obj in objects if obj['category_name'] == target_noun]

        inputs = processor(images=image, text=caption, return_tensors="pt").to(device)

        tokens = processor.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        special_ids = processor.tokenizer.all_special_ids
        valid_indices = [i for i, idx in enumerate(inputs["input_ids"][0].tolist()) if idx not in special_ids]
        target_indices = []
        for i, t in enumerate(tokens):
            if i in valid_indices:
                clean_token = t.lower().replace("##", "")
                if any(word in clean_token or clean_token in word for word in target_noun.lower().split()):
                    target_indices.append(i)
        if not target_indices:
            continue
        tgt_rel = [valid_indices.index(idx) for idx in target_indices]

        heatmaps = extract_all_heatmaps(inputs, valid_indices, tgt_rel)
        if heatmaps is None:
            continue

        for m_name in methods:
            hm = heatmaps[m_name]
            if hm.max() == 0:
                continue
            max_idx = np.argmax(hm)
            max_y_patch, max_x_patch = np.unravel_index(max_idx, hm.shape)
            rel_x = (max_x_patch + 0.5) / GRID_SIZE
            rel_y = (max_y_patch + 0.5) / GRID_SIZE
            pixel_x = rel_x * orig_W
            pixel_y = rel_y * orig_H
            hit = False
            for bbox in all_target_bboxes:
                x_min, y_min, bbox_w, bbox_h = bbox
                x_max, y_max = x_min + bbox_w, y_min + bbox_h
                if (x_min <= pixel_x <= x_max) and (y_min <= pixel_y <= y_max):
                    hit = True
                    break
            if hit:
                hits[m_name] += 1
            results_per_sample[m_name].append(1 if hit else 0)

        evaluated += 1
        pbar.update(1)
        if evaluated % 100 == 0:
            pbar.write(f"\n--- Pointing Game Accuracy ({evaluated}/{NUM_EVALS}) ---")
            for m_name in methods:
                acc = (hits[m_name] / evaluated) * 100
                pbar.write(f"{m_name:<25} | {acc:.1f}%")
            pbar.write("-" * 50)


print("\n" + "=" * 60)
print(" FINAL FLICKR30K POINTING GAME ACCURACY (Phrase Grounding) ".center(60))
print("=" * 60)
print(f"{'Method':<25} | {'Hit Acc (%)':<12} | {'Variance':<10} | {'Std Dev':<10}")
print("-" * 60)
for m_name in methods:
    arr = np.array(results_per_sample[m_name])
    acc = arr.mean() * 100 if len(arr) > 0 else 0
    var = arr.var(ddof=1) if len(arr) > 1 else 0.0
    std = arr.std(ddof=1) if len(arr) > 1 else 0.0
    prefix = ">> " if "Deep-Spec" in m_name else "   "
    print(f"{prefix}{m_name:<22} | {acc:>10.2f} | {var:>10.4f} | {std:>10.4f}")
print("=" * 60)

Loading BLIP Foundation Model...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

Loaded annotations for 999 test images.
Starting Golden Run: Large-Scale BLIP Pointing Game over 1000 Flickr30k Entities...



Evaluating:  10%|█         | 100/1000 [01:31<13:22,  1.12obj/s]


--- Pointing Game Accuracy (100/1000) ---
Raw Attention             | 16.0%
Attention Rollout         | 16.0%
ViT-Grad-CAM              | 26.0%
Attn x Grad (Chefer)      | 37.0%
Deep-Spec (Ours)          | 60.0%
--------------------------------------------------


...


--- Pointing Game Accuracy (700/1000) ---
Raw Attention             | 16.4%
Attention Rollout         | 14.4%
ViT-Grad-CAM              | 22.1%
Attn x Grad (Chefer)      | 37.0%
Deep-Spec (Ours)          | 59.6%
--------------------------------------------------


Evaluating:  80%|████████  | 800/1000 [12:26<02:55,  1.14obj/s]


--- Pointing Game Accuracy (800/1000) ---
Raw Attention             | 16.2%
Attention Rollout         | 14.5%
ViT-Grad-CAM              | 21.9%
Attn x Grad (Chefer)      | 36.2%
Deep-Spec (Ours)          | 59.6%
--------------------------------------------------


Evaluating:  87%|████████▋ | 872/1000 [13:32<01:59,  1.07obj/s]


 FINAL FLICKR30K POINTING GAME ACCURACY (Phrase Grounding)  
Method                    | Hit Acc (%)  | Variance   | Std Dev   
------------------------------------------------------------
   Raw Attention          |      16.74 |     0.1396 |     0.3736
   Attention Rollout      |      15.02 |     0.1278 |     0.3575
   ViT-Grad-CAM           |      22.50 |     0.1746 |     0.4178
   Attn x Grad (Chefer)   |      36.81 |     0.2329 |     0.4826
>> Deep-Spec (Ours)       |      60.09 |     0.2401 |     0.4900
